# 파일 불러오기

In [14]:
# !pip install pypdf

In [15]:
import json
import re
import time
from pathlib import Path
from typing import List, Dict, Any

from openai import OpenAI

In [16]:
from google.colab import userdata
import os

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

In [17]:
# 1. 기본 설정
MODEL_NAME = "gpt-4.1-mini"

INPUT_FILES = [
    "./data/f1_glossary_all.json",
    "./data/f1_history_wiki.json",
]

OUTPUT_DIR = Path("output_qa_pipeline")
OUTPUT_DIR.mkdir(exist_ok=True)

FINETUNE_JSONL_PATH = OUTPUT_DIR / "f1_finetune_qa.jsonl"

SLEEP_SECONDS = 0.5
MAX_RETRIES = 3

client = OpenAI()

In [18]:
def load_json_as_articles(path: str) -> List[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    source_file = Path(path).name
    articles = []

    for item in data:
        if "term" in item:  # glossary 스키마
            term = item.get("term", "").strip()
            desc = item.get("description", "").strip()
            example = item.get("example", "").strip()
            article_text = f"용어: {term}\n설명: {desc}"
            if example:
                article_text += f"\n예시: {example}"
            articles.append({
                "source_file": source_file,
                "article_id": f"GLOSS.{re.sub(r'[^A-Za-z0-9]', '_', term)}",
                "article_title": term,
                "article_text": article_text,
            })

        elif "content" in item:  # history 스키마
            title = item.get("title", "").strip()
            content = item.get("content", "").strip()
            safe_id = re.sub(r'[^A-Za-z0-9]', '_', title)
            articles.append({
                "source_file": source_file,
                "article_id": f"HIST.{safe_id}",
                "article_title": title,
                "article_text": f"{title}\n\n{content}",
            })

    return articles

def load_multiple_jsons(paths):
    articles = []
    for path in paths:
        loaded = load_json_as_articles(path)
        print(f"  - {Path(path).name}: {len(loaded)}개 항목")
        articles.extend(loaded)
    return articles

In [19]:
# 3. 텍스트 정리 + 문장 분리
def normalize_markdown(text: str) -> str:
    # 페이지 푸터 제거: "> 27 February 2026 D4 Issue 05"
    text = re.sub(r'>\s*\d+ \w+ \d+ [A-Z]\d+ Issue \d+\s*\n', '', text)
    # 페이지 헤더 제거: "SECTION D: FINANCIAL REGULATIONS..."
    text = re.sub(r'SECTION [A-Z]: .+?\n', '', text)
    # **볼드** 제거
    text = re.sub(r'\*\*(.+?)\*\*', r'\1', text)
    # _이탤릭_ 제거
    text = re.sub(r'_(.+?)_', r'\1', text)
    # ## 헤더 기호 제거
    text = re.sub(r'^#{1,6}\s+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


def split_articles(text: str, source_file: str) -> List[Dict[str, Any]]:
    text = normalize_text(text)

    # ✅ 수정 1: 앞 공백 허용 + 점 없는 상위 섹션도 포함
    raw_header_pattern = re.compile(
        r'^(#{1,3})\s+\**\s*([A-Z]\d+(?:\.\d+){0,3})[^\n]*?\**\s*$',
        re.MULTILINE
    )
    matches = list(pattern.finditer(text))

    articles = []

    for i, match in enumerate(matches):
        article_id = match.group(1).strip()
        article_title = match.group(2).strip()

        start = match.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        article_text = text[start:end].strip()

        # ✅ 수정 2: 너무 짧은 것 제외 (기존)
        if len(article_text) < 100:
            continue

        # ✅ 수정 3: [Replaced by...] 단독 조문 제외
        body = article_text[len(match.group(0)):].strip()
        if re.match(r'^\[Replaced by', body) and len(body) < 80:
            continue

        # ✅ 수정 4: 목차 제외 (한 조문 안에 조문 번호가 5개 이상 나열된 경우)
        sub_article_count = len(re.findall(r'[A-Z]\d+\.\d+', article_text))
        if sub_article_count > 8 and '\n' not in article_text[:200]:
            continue

        articles.append({
            "source_file": source_file,
            "article_id": article_id,
            "article_title": article_title,
            "article_text": article_text
        })

    return articles

In [20]:
def split_articles(articles, source_file):
    return [a for a in articles if a["source_file"] == source_file]

In [21]:
def split_definitions(articles, source_file):
    return []

In [22]:
# 4. Q & A 생성
def build_prompt(article: Dict[str, Any]) -> str:
    return f"""
너는 F1 입문자용 규정 설명 챗봇의 파인튜닝 데이터를 만드는 도우미야.

아래 규정 조문만 보고 한국어 질문-답변 데이터를 만들어.
규정에 없는 내용은 추측하지 마.
숫자, 조건, 예외는 원문 기준으로 유지해.

[조문 정보]
- 파일명: {article["source_file"]}
- 조문 ID: {article["article_id"]}
- 조문 제목: {article["article_title"]}

[조문 원문]
{article["article_text"]}

[생성 규칙]
- 조문 내용만 바탕으로 작성
- 한국어로 작성
- 초보자도 이해하기 쉽게 설명
- 답변은 6  문장
- 숫자, 단위(mm, ms, 개수, 각도)는 조문 기준 그대로 유지
- 규정 원문에 없는 내용 단정 금지
- 질문끼리 의미 중복이 크지 않게 작성
- 오해 교정형은 왜 틀렸는지도 설명할 것
- 비교형은 같은 조문 안에서 비교 가능한 대상이 있을 때만 만들 것
- 단순 문장 재진술만 하지 말고, 입문자 질문처럼 자연스럽게 만들 것
- 조항 번호만 언급하지 말고, 내용을 풀어서 설명할 것
- 아래 유형을 고루 섞기:
1. 개념형
2. 요약형
3. 수치/기준형
4. 오해 교정형

[출력 형식]
반드시 아래 JSON 배열만 출력:
[
  {{
    "question": "...",
    "answer": "..."
  }}
]
""".strip()


def extract_json_array(text: str) -> str:
    match = re.search(r"\[.*\]", text, re.DOTALL)
    if not match:
        raise ValueError("JSON 배열을 찾지 못했습니다.")
    return match.group(0)


def generate_qa(article: Dict[str, Any]) -> List[Dict[str, Any]]:
    prompt = build_prompt(article)

    for attempt in range(MAX_RETRIES):
        try:
            response = client.responses.create(
                model=MODEL_NAME,
                input=prompt
            )

            text = response.output_text.strip()

            try:
                data = json.loads(text)
            except json.JSONDecodeError:
                data = json.loads(extract_json_array(text))

            results = []
            for item in data:
                if not isinstance(item, dict):
                    continue
                if "question" not in item or "answer" not in item:
                    continue

                results.append({
                    "source_file": article["source_file"],
                    "article_id": article["article_id"],
                    "article_title": article["article_title"],
                    "question": item["question"],
                    "answer": item["answer"]
                })

            return results

        except Exception as e:
            print(f"[재시도 {attempt + 1}/{MAX_RETRIES}] {article['article_id']} 실패: {e}")
            time.sleep(1.5)

    return []

In [23]:
# 5. 중복 제거
def normalize_question(q: str) -> str:
    q = q.strip().lower()
    q = re.sub(r"[^\w가-힣\s]", "", q)
    q = re.sub(r"\s+", " ", q)
    return q


def deduplicate_qa(data: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    seen = set()
    deduped = []

    for item in data:
        key = (
            normalize_question(item["question"]),
            re.sub(r"\s+", " ", item["answer"].strip().lower())
        )

        if key in seen:
            continue

        seen.add(key)
        deduped.append(item)

    return deduped

In [24]:
def to_finetune_messages(item):
    return {
        "messages": [
            {
                "role": "user",
                "content": item["question"]
            },
            {
                "role": "assistant",
                "content": item["answer"]
            }
        ]
    }

---

In [25]:
def save_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

---

In [26]:
def main():
    print("[1] JSON 파일 불러오기")
    all_articles = load_multiple_jsons(INPUT_FILES)

    print(f"\n총 조문 수: {len(all_articles)}")  # ← articles → all_articles

    print("\n[2] QA 생성")
    all_qa = []
    for i, article in enumerate(all_articles, start=1):  # ← articles → all_articles
        print(f"  - ({i}/{len(all_articles)}) [{article['article_id']}] {article['article_title'][:40]}")  # ← 여기도
        qa_items = generate_qa(article)
        all_qa.extend(qa_items)
        time.sleep(SLEEP_SECONDS)

    print(f"\n생성된 QA 수: {len(all_qa)}")

    print("[3] 중복 제거")
    deduped = deduplicate_qa(all_qa)
    print(f"중복 제거 후 QA 수: {len(deduped)}")

    print("[4] JSONL 저장")
    rows = [to_finetune_messages(item) for item in deduped]
    save_jsonl(FINETUNE_JSONL_PATH, rows)

    print("\n완료")
    print(f"저장 위치: {FINETUNE_JSONL_PATH}")


if __name__ == "__main__":
    main()

[1] JSON 파일 불러오기
  - f1_glossary_all.json: 244개 항목
  - f1_history_wiki.json: 23개 항목

총 조문 수: 267

[2] QA 생성
  - (1/267) [GLOSS.ACTIVE_AERO] ACTIVE AERO
  - (2/267) [GLOSS.ADVANCED_SUSTAINABLE_FUEL] ADVANCED SUSTAINABLE FUEL
  - (3/267) [GLOSS.AERO_RAKE] AERO RAKE
  - (4/267) [GLOSS.AERODYNAMICS] AERODYNAMICS
  - (5/267) [GLOSS.AIR_INTAKE] AIR INTAKE
  - (6/267) [GLOSS.ALLOCATION] ALLOCATION
  - (7/267) [GLOSS.APEX] APEX
  - (8/267) [GLOSS.APPEAL] APPEAL
  - (9/267) [GLOSS.AQUAPLANING] AQUAPLANING
  - (10/267) [GLOSS.ARMCO] ARMCO
  - (11/267) [GLOSS.ASPHALT] ASPHALT
  - (12/267) [GLOSS.BACKMARKER] BACKMARKER
  - (13/267) [GLOSS.BALLAST] BALLAST
  - (14/267) [GLOSS.BARGEBOARD] BARGEBOARD
  - (15/267) [GLOSS.BATTERY] BATTERY
  - (16/267) [GLOSS.BLACK_FLAG] BLACK FLAG
  - (17/267) [GLOSS.BLACK_AND_ORANGE_FLAG] BLACK AND ORANGE FLAG
  - (18/267) [GLOSS.BLACK_AND_WHITE_FLAG] BLACK AND WHITE FLAG
  - (19/267) [GLOSS.BLISTERING] BLISTERING
  - (20/267) [GLOSS.BLUE_FLAG] BLUE FLAG
  - (21/267) 